### 생성 데이터 컬럼
> 위치 및 충전기 정보
- 충전소위치  충전기 <br/>	
> 충전기 정보 상세
- 충전기타입  충전기상태  충전기용량  방전지원여부  충전량  요청충전량  방전량  요청방전량<br>  
> 시간 정보
- 연결시작시간  연결종료시간  충전시작시간  충전종료시간  출방예상시간<br/>
> 차량 Soc 정보
- 베터리용량  시작값  종료값  연비

In [1]:
import pandas as pd
import numpy as np
import random
import json
from datetime import datetime, timedelta, timezone
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy.stats import gaussian_kde

plt.rcParams['font.family'] ='Malgun Gothic'
plt.rcParams['axes.unicode_minus'] =False

# 시드 고정
np.random.seed(42)
random.seed(42)

In [ ]:
# 파일 로딩
locations_df = pd.read_csv('stat_id.csv', header=None)           # 충전소 코드
store_num_df = pd.read_csv('charger_num.csv', header=None, names=['id', 'num'])           # 충전소별 충전기 개수
type_df = pd.read_csv('connector_types.csv', header=None, names=['code', 'type'])  # 타입 정보

store_num_dict = dict(zip(store_num_df['id'], store_num_df['num']))
type_dict = type_df.groupby('code')['type'].apply(list).to_dict()
locations = locations_df.loc[0].tolist()

# 충전기 정보 저장
evse_dict = defaultdict(list)
evse_info = {}

fc_evse_volume_list = [50, 100, 200]
sc_evse_volume_list = [10, 30, 50]

for idx, location in enumerate(locations):
    if location not in store_num_dict or pd.isna(store_num_dict[location]):
        continue
    evse_n = int(store_num_dict[location])
    for i in range(evse_n):
        evse_name = f'{str(i).zfill(2)}'
        evse_key = f'{location}_{evse_name}'
        evse_dict[location].append(evse_name)

        if store_num_dict[location] in type_dict:
            evse_type = random.choice(['SC'] if location == 'AC완속' else ['FC'])
        else:
            evse_type = random.choice(['FC', 'SC'])

        volume = random.choice(fc_evse_volume_list if evse_type == 'FC' else sc_evse_volume_list)
        evse_info[evse_key] = {'type': evse_type, 'volume': volume}

# 기타 설정
yes_no = ['y', 'n']
battery_capacities = [50, 60, 75, 90, 100]
now = datetime.now()
TIME_LIMIT = now + timedelta(days=13)

evse_popularity_score = {}
evse_last_end_time = {}

# evse_last_end_time 초기화 (시뮬레이션 시작 시점으로 과거 날짜 지정)
start_reference_time = now - timedelta(days=365)
for location, evse_names in evse_dict.items():
    for evse_name in evse_names:
        evse_key = f"{location}_{evse_name}"
        evse_last_end_time[evse_key] = start_reference_time

# 함수 정의
def generate_battery_percent(start=True, delivered_energy=None, capacity=100):
    if start:
        return round(np.clip(np.random.beta(2, 5) * 100, 5, 60), 1)
    else:
        if delivered_energy is None:
            return round(np.clip(np.random.beta(5, 2) * 100, 30, 100), 1)
        increase = min((delivered_energy / capacity) * 100 + np.random.normal(0, 3), 100)
        return round(np.clip(increase, 5, 100), 1)

def generate_requested_energy():
    if np.random.rand() < 0.008:
        return 0.0
    val = np.random.lognormal(mean=2.98, sigma=0.4)
    return round(val, 2)

def generate_energy_usage(evse_volume, request_energy):
    if request_energy == 0:
        return round(random.uniform(0, 10), 2) if np.random.rand() < 0.2 else 0.0
    offset = request_energy * 0.037
    return round(random.uniform(request_energy - offset, request_energy + offset), 2)

def generate_discharge(supports_discharge):
    if supports_discharge == 'n':
        return 0, 0
    discharged = int(np.random.gamma(2, 5))
    requested = discharged + random.randint(-3, 5)
    return max(discharged, 0), max(requested, 0)

def generate_charge_duration(evse_type, delivered_energy):
    if delivered_energy == 0:
        return timedelta(minutes=0)
    if evse_type == 'FC':
        charge_speed = random.uniform(50, 80)
        soft_min_minutes = 5
    else:
        charge_speed = random.uniform(3.3, 7.2)
        soft_min_minutes = 10
    charge_time_minutes = (delivered_energy / charge_speed) * 60
    charge_time_minutes *= np.random.normal(1.05, 0.05)
    alpha = np.clip((charge_time_minutes - soft_min_minutes) / soft_min_minutes, 0, 1)
    adjusted_minutes = (1 - alpha) * soft_min_minutes + alpha * charge_time_minutes
    jitter = np.random.normal(0, 2)
    final_minutes = round(np.clip(adjusted_minutes + jitter, soft_min_minutes, 180))
    return timedelta(minutes=final_minutes)

def get_evse_popularity(evse_key):
    if evse_key not in evse_popularity_score:
        evse_popularity_score[evse_key] = np.clip(np.random.beta(2, 3), 0.05, 0.95)
    return evse_popularity_score[evse_key]

def generate_idle_gap_with_demand(evse_key, current_time):
    popularity = get_evse_popularity(evse_key)
    base_gap = 600 - int(popularity * 550)
    hour = current_time.hour
    if 7 <= hour <= 9 or 17 <= hour <= 19:
        demand_factor = 0.8
    elif 1 <= hour <= 5:
        demand_factor = 0.2
    else:
        demand_factor = 0.5
    no_visit_prob = max(0, 1 - popularity * demand_factor)
    if random.random() < no_visit_prob:
        return base_gap + random.randint(6*60, 3*24*60)
    else:
        noise = random.randint(-20, 20)
        return max(base_gap + noise, 10)

def generate_times(evse_type, evse_key, requested_energy, delivered_energy):

    

    last_end = evse_last_end_time.get(evse_key, TIME_LIMIT)
    idle_gap_minutes = generate_idle_gap_with_demand(evse_key, last_end)
    start = last_end + timedelta(minutes=idle_gap_minutes)
    
    if start > TIME_LIMIT:
        raise StopIteration
    
    charge_delay = timedelta(minutes=random.randint(1, 30))
    charge_duration = generate_charge_duration(evse_type, delivered_energy)
    extra_seconds = np.clip(np.random.lognormal(mean=7.5, sigma=0.7), 300, 36000)
    conn_duration = charge_delay + charge_duration + timedelta(seconds=round(extra_seconds))

    chg_start = start + charge_delay if requested_energy > 0 else None
    chg_end = chg_start + charge_duration if chg_start else None
    conn_end = start + conn_duration

    evse_last_end_time[evse_key] = conn_end
    return start, conn_end, chg_start, chg_end, last_end

def generate_departure(chg_start, chg_end):
    if chg_end is None or random.random() < 0.15:
        return None
    if random.random() < 0.8:
        choice = random.random()
        if choice < 0.6:
            delay = np.random.normal(20, 10)
        elif choice < 0.85:
            delay = np.random.exponential(30)
        else:
            delay = np.random.normal(45, 15)
        delay = round(np.clip(delay, 1, 120))
        return chg_end + timedelta(minutes=delay)
    else:
        for _ in range(10):
            if random.random() < 0.7:
                advance = np.random.normal(10, 5)
            else:
                advance = np.random.exponential(15)
            advance = round(np.clip(advance, 1, 60))
            proposed = chg_end - timedelta(minutes=advance)
            if proposed >= chg_start:
                return proposed
        return chg_start + timedelta(minutes=1)

# 🔼 데이터 생성
start_date = datetime.now() - timedelta(days=365)
end_date = TIME_LIMIT
days = (end_date - start_date).days + 1
n_sessions_per_day = 48  # 하루 30분 간격

def generate_fixed_dataframe(evse_dict, days, n_sessions_per_day):
    data = []
    for location, evse_list in evse_dict.items():
        for evse_name in evse_list:
            evse_key = f'{location}_{evse_name}'
            evse_type = evse_info[evse_key]['type']
            evse_volume = evse_info[evse_key]['volume']
            for day_idx in range(days):
                for session_idx in range(n_sessions_per_day):
                    # 1️⃣ 요청 에너지와 실제 충전량
                    requested_energy = generate_requested_energy()
                    delivered_energy = generate_energy_usage(evse_volume, requested_energy)
                    # 2️⃣ 시계열 값 생성
                    try:
                        start, conn_end, chg_start, chg_end, prev_end = generate_times(
                            evse_type, evse_key, requested_energy, delivered_energy
                        )
                    except StopIteration:
                        continue
                    # 3️⃣ 배터리 퍼센트
                    battery_capacity = np.random.choice(battery_capacities)
                    battery_start = generate_battery_percent(start=True, capacity=battery_capacity)
                    battery_end = generate_battery_percent(
                        start=False, delivered_energy=delivered_energy, capacity=battery_capacity
                    )
                    # 4️⃣ 방전 여부/양
                    supports_discharge = 'y' if evse_type == 'SC' and random.random() < 0.5 else 'n'
                    discharge, requested_discharge = generate_discharge(supports_discharge)
                    # 5️⃣ 예약여부
                    scheduled_charge = random.choices(['y','n'], weights=[0.3,0.7])[0]
                    # 6️⃣ 효율 (연비)
                    efficiency = round(np.clip(np.random.normal(5.5, 0.8), 3.5, 7.5), 2)
                    # 7️⃣ 출발예상시간 (출발전 랜덤 로직 적용)
                    departure_time = generate_departure(chg_start, chg_end)
                    if departure_time is not None:
                        departure_time_str = departure_time.isoformat()
                    else:
                        # None이면 연결종료 3분 후(기존 로직 fallback)
                        departure_time_str = (conn_end + timedelta(minutes=3)).isoformat()
                    # 8️⃣ row 생성
                    data.append({
                        '충전소위치': location,
                        '충전기이름': evse_name,
                        '충전기타입': evse_type,
                        '충전기상태': 'Available',
                        '충전기용량': evse_volume,
                        '방전지원여부': supports_discharge,
                        '예약충전': scheduled_charge,
                        '충전량(kWh)': delivered_energy,
                        '요청충전량(kWh)': requested_energy,
                        '방전량(kWh)': discharge,
                        '요청방전량(kWh)': requested_discharge,
                        '마지막충전종료시간': prev_end.isoformat(),
                        '연결시작시간': start.isoformat(),
                        '충전시작시간': chg_start.isoformat() if chg_start else "",
                        '충전종료시간': chg_end.isoformat() if chg_end else "",
                        '연결종료시간': conn_end.isoformat(),
                        '출발예상시간': departure_time_str,
                        '베터리용량(kWh)': battery_capacity,
                        '시작베터리%': battery_start,
                        '종료베터리%': battery_end,
                        '연비(Wh/km)': efficiency
                    })
    return pd.DataFrame(data)


print(evse_dict)

# 🔽 실행
df_clean = generate_fixed_dataframe(evse_dict, days, n_sessions_per_day)
df_clean.head()
df_clean.to_csv('50area_dummy1_232,000.csv',index=False)


defaultdict(<class 'list'>, {'CSCS2015': ['00'], 'CV000666': ['00', '01'], 'CV000821': ['00'], 'CV001074': ['00', '01', '02', '03'], 'CV001664': ['00', '01', '02'], 'CV001665': ['00', '01', '02', '03'], 'CV001718': ['00', '01'], 'CV003321': ['00', '01'], 'CV003367': ['00', '01'], 'CV003581': ['00', '01'], 'CV003599': ['00', '01'], 'CV003600': ['00', '01'], 'CV003608': ['00', '01', '02', '03'], 'CV003609': ['00', '01'], 'CV003656': ['00', '01', '02', '03'], 'CV003675': ['00', '01', '02', '03', '04'], 'CV003677': ['00', '01', '02', '03', '04'], 'CV003744': ['00', '01', '02', '03'], 'CV003782': ['00', '01', '02', '03', '04', '05', '06', '07', '08'], 'CV003794': ['00', '01'], 'CV003806': ['00', '01'], 'CV004087': ['00'], 'CV004239': ['00', '01'], 'CV004240': ['00', '01'], 'DL000080': ['00'], 'DO000038': ['00'], 'EC000206': ['00', '01'], 'EC004763': ['00', '01', '02', '03'], 'EC005014': ['00'], 'EO000254': ['00'], 'EP200258': ['00'], 'EREL0040': ['00', '01', '02', '03', '04', '05'], 'EREL00

OSError: Cannot save file into a non-existent directory: '..\csv'

In [ ]:
df_clean.head()

,충전소위치,충전기이름,충전기타입,충전기상태,충전기용량,방전지원여부,예약충전,충전량(kWh),요청충전량(kWh),방전량(kWh),...,마지막충전종료시간,연결시작시간,충전시작시간,충전종료시간,연결종료시간,출발예상시간,베터리용량(kWh),시작베터리%,종료베터리%,연비(Wh/km)
0,CSCS2015,00,FC,Available,50,n,n,12.84,12.62,0,...,2024-07-28T11:46:02.688864,2024-07-30T00:23:02.688864,2024-07-30T00:40:02.688864,2024-07-30T00:50:02.688864,2024-07-30T01:10:54.688864,2024-07-30T00:51:02.688864,100,33.0,5.5,5.98
1,CSCS2015,00,FC,Available,50,n,n,17.64,17.81,0,...,2024-07-30T01:10:54.688864,2024-07-31T19:16:54.688864,2024-07-31T19:35:54.688864,2024-07-31T19:51:54.688864,2024-07-31T21:17:14.688864,2024-07-31T20:05:54.688864,90,17.7,17.8,5.27
2,CSCS2015,00,FC,Available,50,n,n,40.58,41.30,0,...,2024-07-31T21:17:14.688864,2024-08-03T15:56:14.688864,2024-08-03T16:15:14.688864,2024-08-03T16:52:14.688864,2024-08-03T17:17:54.688864,2024-08-03T16:55:14.688864,90,14.0,48.7,6.09
3,CSCS2015,00,FC,Available,50,n,y,21.27,21.08,0,...,2024-08-03T17:17:54.688864,2024-08-04T00:46:54.688864,2024-08-04T01:13:54.688864,2024-08-04T01:31:54.688864,2024-08-04T01:42:36.688864,2024-08-04T01:46:54.688864,100,15.6,22.5,4.83
4,CSCS2015,00,FC,Available,50,n,n,9.17,9.30,0,...,2024-08-04T01:42:36.688864,2024-08-06T14:30:36.688864,2024-08-06T14:37:36.688864,2024-08-06T14:46:36.688864,2024-08-06T15:02:34.688864,2024-08-06T14:40:36.688864,75,16.6,10.8,5.35


In [ ]:
df_clean.duplicated(keep=False).sum()

np.int64(0)

: 

: 

: 

: 

: 

In [ ]:
start = pd.to_datetime(df_clean['연결시작시간'],format='ISO8601')
last_end = pd.to_datetime(df_clean['마지막충전종료시간'],format='ISO8601')

delta = start-last_end
mask = delta > pd.Timedelta(days=0)
delta

KeyError: '연결시작시간'

In [ ]:
df = df_clean[['충전기타입','요청충전량(kWh)','충전시작시간','충전종료시간']]
df[df['요청충전량(kWh)'].between(10, 15)]

,충전기타입,요청충전량(kWh),충전시작시간,충전종료시간
0,FC,12.62,2024-07-02T12:54:00+00:00,2024-07-02T13:04:00+00:00
12,FC,11.05,2024-07-23T21:45:17+00:00,2024-07-23T21:58:17+00:00
30,FC,12.32,2024-08-25T06:07:40+00:00,2024-08-25T06:13:40+00:00
42,FC,12.88,2024-09-14T05:06:57+00:00,2024-09-14T05:19:57+00:00
45,FC,14.21,2024-09-19T10:15:26+00:00,2024-09-19T10:31:26+00:00
...,...,...,...,...
244834,SC,10.91,2025-06-30T08:27:00+00:00,2025-06-30T10:30:00+00:00
244837,SC,11.48,2025-07-02T14:41:46+00:00,2025-07-02T17:41:46+00:00
244839,SC,14.38,2025-07-07T07:37:04+00:00,2025-07-07T10:37:04+00:00
244840,SC,14.57,2025-07-09T10:01:26+00:00,2025-07-09T13:01:26+00:00


: 

: 

: 

: 

: 

In [ ]:
df_clean['출발예상시간']>df_clean['연결시작시간']

0         True
1         True
2         True
3         True
4         True
          ... 
244847    True
244848    True
244849    True
244850    True
244851    True
Length: 244852, dtype: bool

: 

: 

: 

: 

: 

In [ ]:
mask = (df_clean['출발예상시간']>df_clean['연결시작시간'])==False
df_clean[mask]

,충전소위치,충전기이름,충전기타입,충전기상태,충전기용량,방전지원여부,예약충전,충전량(kWh),요청충전량(kWh),방전량(kWh),...,마지막충전종료시간,연결시작시간,충전시작시간,충전종료시간,연결종료시간,출발예상시간,베터리용량(kWh),시작베터리%,종료베터리%,연비(Wh/km)


: 

: 

: 

: 

: 

In [ ]:
df_clean['마지막충전종료시간']<df_clean['연결시작시간']

0         True
1         True
2         True
3         True
4         True
          ... 
244847    True
244848    True
244849    True
244850    True
244851    True
Length: 244852, dtype: bool

: 

: 

: 

: 

: 

In [ ]:
df_clean['마지막충전종료시간']<df_clean['연결종료시간']

0         True
1         True
2         True
3         True
4         True
          ... 
244847    True
244848    True
244849    True
244850    True
244851    True
Length: 244852, dtype: bool

: 

: 

: 

: 

: 

In [ ]:
df_clean['연결종료시간']>df_clean['연결시작시간']

0         True
1         True
2         True
3         True
4         True
          ... 
244847    True
244848    True
244849    True
244850    True
244851    True
Length: 244852, dtype: bool

: 

: 

: 

: 

: 

In [ ]:
mask = (df_clean['충전종료시간']>df_clean['충전시작시간']) == False
df_clean[mask]

,충전소위치,충전기이름,충전기타입,충전기상태,충전기용량,방전지원여부,예약충전,충전량(kWh),요청충전량(kWh),방전량(kWh),...,마지막충전종료시간,연결시작시간,충전시작시간,충전종료시간,연결종료시간,출발예상시간,베터리용량(kWh),시작베터리%,종료베터리%,연비(Wh/km)
18,CSCS2015,00,FC,Available,50,n,n,0.00,0.0,0,...,2024-08-01T16:58:32+00:00,2024-08-03T02:46:32+00:00,,,2024-08-03T03:40:05+00:00,2024-08-03T03:43:05+00:00,90,36.1,5.0,4.30
74,CSCS2015,00,FC,Available,50,n,n,0.00,0.0,0,...,2024-11-11T11:13:52+00:00,2024-11-13T14:26:52+00:00,,,2024-11-13T14:58:59+00:00,2024-11-13T15:01:59+00:00,100,21.2,5.0,3.50
77,CSCS2015,00,FC,Available,50,n,n,0.00,0.0,0,...,2024-11-16T19:12:31+00:00,2024-11-18T12:13:31+00:00,,,2024-11-18T13:45:16+00:00,2024-11-18T13:48:16+00:00,75,25.9,5.0,5.67
103,CSCS2015,00,FC,Available,50,n,n,0.00,0.0,0,...,2025-01-05T10:11:22+00:00,2025-01-07T18:42:22+00:00,,,2025-01-07T21:55:27+00:00,2025-01-07T21:58:27+00:00,60,50.2,5.2,5.28
178,CSCS2015,00,FC,Available,50,n,n,0.00,0.0,0,...,2025-05-26T11:08:04+00:00,2025-05-29T16:43:04+00:00,,,2025-05-29T17:45:56+00:00,2025-05-29T17:48:56+00:00,100,48.8,5.0,7.18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
244087,ST600547,01,SC,Available,10,y,n,0.00,0.0,19,...,2025-04-15T13:59:36+00:00,2025-04-16T07:17:36+00:00,,,2025-04-16T07:40:29+00:00,2025-04-16T07:43:29+00:00,90,17.2,5.0,6.33
244174,ST600547,02,FC,Available,100,n,y,3.58,0.0,0,...,2024-07-18T17:28:55+00:00,2024-07-19T01:00:55+00:00,,,2024-07-19T01:20:29+00:00,2024-07-19T01:23:29+00:00,100,44.8,8.8,6.10
244404,SZAA0032,00,SC,Available,10,n,y,0.00,0.0,0,...,2024-07-26T12:49:33+00:00,2024-07-28T02:47:33+00:00,,,2024-07-28T03:55:34+00:00,2024-07-28T03:58:34+00:00,60,12.8,5.0,6.03
244478,SZAA0032,00,SC,Available,10,y,n,0.00,0.0,7,...,2024-11-28T08:19:40+00:00,2024-11-28T15:11:40+00:00,,,2024-11-28T15:35:42+00:00,2024-11-28T15:38:42+00:00,60,16.7,5.0,5.66


: 

: 

: 

: 

: 

In [ ]:
ts = df_clean[mask]
ts['연결시작시간']<ts['연결종료시간']

18        True
74        True
77        True
103       True
178       True
          ... 
244087    True
244174    True
244404    True
244478    True
244758    True
Length: 1998, dtype: bool

: 

: 

: 

: 

: 

In [ ]:
df_clean[df_clean['충전소위치'] == 'PW800657']

,충전소위치,충전기이름,충전기타입,충전기상태,충전기용량,방전지원여부,예약충전,충전량(kWh),요청충전량(kWh),방전량(kWh),...,마지막충전종료시간,연결시작시간,충전시작시간,충전종료시간,연결종료시간,출발예상시간,베터리용량(kWh),시작베터리%,종료베터리%,연비(Wh/km)
233627,PW800657,00,SC,Available,30,n,n,18.64,19.23,0,...,2024-07-01T00:00:00+00:00,2024-07-03T07:35:00+00:00,2024-07-03T08:02:00+00:00,2024-07-03T11:02:00+00:00,2024-07-03T11:54:38+00:00,2024-07-03T11:30:00+00:00,100,60.0,16.8,5.17
233628,PW800657,00,SC,Available,30,n,y,17.56,17.57,0,...,2024-07-03T11:54:38+00:00,2024-07-05T13:28:38+00:00,2024-07-05T13:39:38+00:00,2024-07-05T16:39:38+00:00,2024-07-05T17:25:36+00:00,2024-07-05T17:28:36+00:00,60,44.8,27.9,7.33
233629,PW800657,00,SC,Available,30,y,y,21.31,21.32,8,...,2024-07-05T17:25:36+00:00,2024-07-06T07:51:36+00:00,2024-07-06T07:56:36+00:00,2024-07-06T10:56:36+00:00,2024-07-06T11:16:07+00:00,2024-07-06T11:19:07+00:00,60,5.1,37.7,4.58
233630,PW800657,00,SC,Available,30,n,n,23.30,22.87,0,...,2024-07-06T11:16:07+00:00,2024-07-09T08:16:07+00:00,2024-07-09T08:38:07+00:00,2024-07-09T11:38:07+00:00,2024-07-09T13:02:11+00:00,2024-07-09T13:05:11+00:00,100,40.9,23.9,6.59
233631,PW800657,00,SC,Available,30,y,y,21.16,20.48,18,...,2024-07-09T13:02:11+00:00,2024-07-09T20:42:11+00:00,2024-07-09T20:47:11+00:00,2024-07-09T23:47:11+00:00,2024-07-09T23:56:06+00:00,2024-07-10T00:19:11+00:00,60,5.0,35.8,6.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236735,PW800657,13,SC,Available,30,y,y,13.24,12.79,15,...,2025-07-13T17:57:54+00:00,2025-07-14T10:24:54+00:00,2025-07-14T10:54:54+00:00,2025-07-14T13:01:54+00:00,2025-07-14T13:39:30+00:00,2025-07-14T13:24:54+00:00,100,14.4,10.4,6.08
236736,PW800657,13,SC,Available,30,y,n,14.62,14.80,14,...,2025-07-14T13:39:30+00:00,2025-07-14T20:28:30+00:00,2025-07-14T20:35:30+00:00,2025-07-14T22:55:30+00:00,2025-07-14T23:10:12+00:00,2025-07-14T23:13:12+00:00,90,5.0,14.8,4.04
236737,PW800657,13,SC,Available,30,n,n,30.39,30.11,0,...,2025-07-14T23:10:12+00:00,2025-07-16T15:40:12+00:00,2025-07-16T15:59:12+00:00,2025-07-16T18:59:12+00:00,2025-07-16T20:15:05+00:00,2025-07-16T19:35:12+00:00,100,5.0,33.7,5.90
236738,PW800657,13,SC,Available,30,y,n,31.47,31.31,7,...,2025-07-16T20:15:05+00:00,2025-07-20T01:57:05+00:00,2025-07-20T02:16:05+00:00,2025-07-20T05:16:05+00:00,2025-07-20T06:09:26+00:00,2025-07-20T05:30:05+00:00,90,47.7,41.1,7.50


: 

: 

: 

: 

: 

In [ ]:
df_clean[(df_clean['충전소위치'] == 'PW800657') & (df_clean['충전기이름']=='00')]

,충전소위치,충전기이름,충전기타입,충전기상태,충전기용량,방전지원여부,예약충전,충전량(kWh),요청충전량(kWh),방전량(kWh),...,마지막충전종료시간,연결시작시간,충전시작시간,충전종료시간,연결종료시간,출발예상시간,베터리용량(kWh),시작베터리%,종료베터리%,연비(Wh/km)
233627,PW800657,00,SC,Available,30,n,n,18.64,19.23,0,...,2024-07-01T00:00:00+00:00,2024-07-03T07:35:00+00:00,2024-07-03T08:02:00+00:00,2024-07-03T11:02:00+00:00,2024-07-03T11:54:38+00:00,2024-07-03T11:30:00+00:00,100,60.0,16.8,5.17
233628,PW800657,00,SC,Available,30,n,y,17.56,17.57,0,...,2024-07-03T11:54:38+00:00,2024-07-05T13:28:38+00:00,2024-07-05T13:39:38+00:00,2024-07-05T16:39:38+00:00,2024-07-05T17:25:36+00:00,2024-07-05T17:28:36+00:00,60,44.8,27.9,7.33
233629,PW800657,00,SC,Available,30,y,y,21.31,21.32,8,...,2024-07-05T17:25:36+00:00,2024-07-06T07:51:36+00:00,2024-07-06T07:56:36+00:00,2024-07-06T10:56:36+00:00,2024-07-06T11:16:07+00:00,2024-07-06T11:19:07+00:00,60,5.1,37.7,4.58
233630,PW800657,00,SC,Available,30,n,n,23.30,22.87,0,...,2024-07-06T11:16:07+00:00,2024-07-09T08:16:07+00:00,2024-07-09T08:38:07+00:00,2024-07-09T11:38:07+00:00,2024-07-09T13:02:11+00:00,2024-07-09T13:05:11+00:00,100,40.9,23.9,6.59
233631,PW800657,00,SC,Available,30,y,y,21.16,20.48,18,...,2024-07-09T13:02:11+00:00,2024-07-09T20:42:11+00:00,2024-07-09T20:47:11+00:00,2024-07-09T23:47:11+00:00,2024-07-09T23:56:06+00:00,2024-07-10T00:19:11+00:00,60,5.0,35.8,6.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
233824,PW800657,00,SC,Available,30,y,y,28.96,28.91,7,...,2025-07-14T13:51:45+00:00,2025-07-15T15:44:45+00:00,2025-07-15T15:47:45+00:00,2025-07-15T18:47:45+00:00,2025-07-15T19:25:55+00:00,2025-07-15T18:57:45+00:00,75,16.9,38.4,4.98
233825,PW800657,00,SC,Available,30,n,y,17.27,16.80,0,...,2025-07-15T19:25:55+00:00,2025-07-17T22:38:55+00:00,2025-07-17T23:05:55+00:00,2025-07-18T01:54:55+00:00,2025-07-18T02:22:18+00:00,2025-07-18T02:09:55+00:00,100,34.0,22.4,5.17
233826,PW800657,00,SC,Available,30,n,n,24.54,24.23,0,...,2025-07-18T02:22:18+00:00,2025-07-19T01:40:18+00:00,2025-07-19T01:45:18+00:00,2025-07-19T04:45:18+00:00,2025-07-19T05:02:58+00:00,2025-07-19T04:31:18+00:00,100,23.7,20.3,5.36
233827,PW800657,00,SC,Available,30,n,y,25.53,25.33,0,...,2025-07-19T05:02:58+00:00,2025-07-20T05:01:58+00:00,2025-07-20T05:10:58+00:00,2025-07-20T08:10:58+00:00,2025-07-20T09:52:46+00:00,2025-07-20T07:53:58+00:00,90,35.2,24.4,4.25


: 

: 

: 

: 

: 

In [ ]:
df_clean[df_clean['충전소위치']=='st-14']

,충전소위치,충전기이름,충전기타입,충전기상태,충전기용량,방전지원여부,예약충전,충전량(kWh),요청충전량(kWh),방전량(kWh),...,마지막충전종료시간,연결시작시간,충전시작시간,충전종료시간,연결종료시간,출발예상시간,베터리용량(kWh),시작베터리%,종료베터리%,연비(Wh/km)


: 

: 

: 

: 

: 

In [ ]:
df_clean[df_clean['충전기이름']=='st-14_evse-04']

,충전소위치,충전기이름,충전기타입,충전기상태,충전기용량,방전지원여부,예약충전,충전량(kWh),요청충전량(kWh),방전량(kWh),...,마지막충전종료시간,연결시작시간,충전시작시간,충전종료시간,연결종료시간,출발예상시간,베터리용량(kWh),시작베터리%,종료베터리%,연비(Wh/km)


: 

: 

: 

: 

: 

① 방전 지원 여부 → 방전량/요청방전량 가능 여부
     └─ 'n'이면 둘 다 반드시 0
     
② 요청충전량 == 0 → 충전량은 반드시 0 (예외 시 '충전 불필요' 시나리오로 분리)

③ 충전기 상태 → 충전량에 영향
     └─ 'Faulted'이면 충전량=0
     └─ 'Available'인데 충전량 > 0 이면 충돌

④ 충전 시작/종료 시간 → 시간 역전 불가
     └─ 종료 > 시작 > 연결 > 이전 종료

⑤ 시작/종료 배터리 퍼센트 + 충전량 → 일관성 필요
     └─ 종료퍼센트 < 시작퍼센트면 충전량 = 0 or 음수

⑥ 출발예상시간 → 필수 입력 아님
     └─ 사용자 입력 없는 경우 `NaT` 또는 `None`

⑦ 충전기 타입 → 방전지원여부 제약
     └─ 대부분 FC는 방전 미지원 → y 불가



⚡ 충전 상태 기반 패턴 (Status-driven)
- 정상 충전 완료
- 연결 후 충전 실패
- 충전 중 연결 종료
- 중간에 사용자에 의해 중단
- 충전 시작 지연
- 급속 충전 중단
- 완속 충전 연장
- 충전 불필요 (베터리 충분)

🕒 시간 흐름 패턴 (Time-driven) - 일부 반영
- 야간 충전 (22시 ~ 06시)
- 오전 집중 충전 (출근 직전)
- 낮 시간 대기 후 충전
- 급속 충전으로 출발시간 전 도달
- 출발예상시간보다 늦게 충전 종료
- 연결시간 긴데 충전시간 짧음
- 충전기 대기 시간 포함된 시나리오
- 출발예상시간이 예측 충전과 연동
- 베터리 용량 적을 때 반복 충전
- 마지막충전종료시간과 출발시간 간격 비교

🔋 배터리/충전량 기반 패턴 (Energy-driven) -- 아직 반영하지 않음
- 시작 베터리 10%, 종료 90% (충전량 충분)
- 시작 30%, 종료 60% (부분충전)
- 시작 80%, 종료 100% (최적화)
- 종료 후 방전량 존재
- 요청방전량 > 실제방전량
- 방전 후 재충전
- 요청충전량 ≠ 실제충전량 (시간 부족 또는 제약)
- 고용량 충전기에서 30분 만에 완충
- 낮은 연비일 때 충전량 많음
- 시작베터리 높지만 방전 요구 있음

📍 위치/장비 타입 기반 패턴 (Location/type-driven)  -일부 반영
- FC 타입에서 빠른 완충
- SC 타입에서 오래 연결
- 용량이 작지만 빈도 높은 충전기
- FC 충전기에서 방전은 미지원
- SC 충전기에서 연비 반영 충전
- 동일 위치에서 반복 사용되는 사용자
- 충전기이름별 성능 차이 시뮬레이션

🔁 특수 상황 패턴 (Exception-driven) 
- 방전지원 차량만 가능한 시나리오
- 마지막충전종료시간이 기록 누락됨
- 연결시간은 있으나 실제 충전 없음
- 요청 충전량이 0인데 충전됨
- 예측 출발 시간보다 빠르게 종료
